# E6 — Eksperimen tambahan revisi JOIV (Reviewer A & B)

| Bagian | Menjawab | Isi |
|---|---|---|
| **E6c** | B3, B5 | Uji DM (+HLN) semua model terlatih vs random walk, koreksi Holm antar-model |
| **E6d** | B6 | Baterai 6 deret sintetis: 3 tak terprediksi, 3 terprediksi (validitas diskriminan) |
| **E6e** | A11 | Pasar & periode lain (S&P 500, Nikkei, BTC, EUR/USD, 2012–2018) |
| **E6a** | B2 | 5 run GA & PSO independen per ruang pencarian + kurva konvergensi vs anggaran evaluasi |
| **E6b** | B4 | Seleksi bersarang: fitness pada blok yang terpisah dari blok early stopping |

**Cara pakai di Kaggle**
1. Accelerator **GPU T4**, Internet **ON**. Unggah `SIDO_Stock_Data.xlsx` sebagai dataset (opsional; tanpa itu dipakai yfinance).
2. *Save & Run All (Commit)*. Satu commit dibatasi `TIME_BUDGET_H`; pekerjaan yang belum selesai otomatis dilewati.
3. Commit berikutnya: **Add Input → Notebook output** dari versi sebelumnya, lalu commit lagi. Cache dipakai ulang,
   jadi tiap commit melanjutkan. Ulangi sampai baris `Dilewati commit ini: tidak ada`.
4. Opsional: tambahkan juga output notebook E4/E5 sebagai input — evaluasi fitness E5 dan konfigurasi GA/PSO-tight dipakai ulang.

Perkiraan total ±14–18 jam GPU → 2 commit. Urutan: E6c → E6d → E6e → E6a → E6b (yang murah dulu).

In [ ]:
# Yahoo Finance compatibility shim: if yfinance fails (version mismatch such as
# "Impersonating chromeXXX is not supported") or returns nothing, fall back to the Yahoo v8 chart endpoint.
# Prices are dividend/split adjusted, equivalent to auto_adjust=True.
import pandas as pd, requests
try:
    import yfinance as yf
    _yf_download = yf.download
except Exception:
    yf, _yf_download = None, None

def _chart_api(ticker, start, end):
    p1, p2 = int(pd.Timestamp(start).timestamp()), int(pd.Timestamp(end).timestamp())
    for host in ('query1', 'query2'):
        r = requests.get(f'https://{host}.finance.yahoo.com/v8/finance/chart/{ticker}',
                         params=dict(period1=p1, period2=p2, interval='1d', events='div,splits'),
                         headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/124 Safari/537.36'}, timeout=30)
        if r.status_code != 200: continue
        res = r.json()['chart']['result'][0]; q = res['indicators']['quote'][0]
        adj = res['indicators'].get('adjclose', [{}])[0].get('adjclose', q['close'])
        d = pd.DataFrame({'Date': pd.to_datetime(res['timestamp'], unit='s'), 'Open': q['open'], 'High': q['high'],
                          'Low': q['low'], 'Close': q['close'], 'Volume': q['volume'], 'Adj': adj}).dropna()
        f = d['Adj'] / d['Close']
        for c in ('Open', 'High', 'Low', 'Close'): d[c] = d[c] * f
        d['Date'] = (d['Date'] + pd.Timedelta(hours=7)).dt.normalize()
        return d.drop(columns='Adj').set_index('Date')
    raise RuntimeError(f'chart API failed for {ticker}')

def _download(ticker, start=None, end=None, **kw):
    try:
        d = _yf_download(ticker, start=start, end=end, **kw) if _yf_download else None
        if d is not None and len(d) > 200: return d
    except Exception:
        pass
    return _chart_api(ticker, start, end)

if yf is not None: yf.download = _download


In [ ]:
# ============================================================================
# SEL 1. KONFIGURASI, ANGGARAN WAKTU, CACHE
# ============================================================================
import os, glob, math, time, pickle, random, warnings, shutil, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
warnings.filterwarnings('ignore')

T_MULAI       = time.time()
TIME_BUDGET_H = 11.0          # batas commit Kaggle 12 jam; sisakan 1 jam
RUN = dict(E6C=True, E6D=True, E6E=True, E6A=True, E6B=True)

SEED          = 42
N_SEED_E6C    = 15            # seed per model untuk uji DM (E6c)
N_SEED_SYN    = 5             # seed per deret sintetis / pasar (E6d, E6e)
R_SEARCH      = 5             # run pencarian independen per optimizer per ruang (E6a)
R_NESTED      = 3             # run per optimizer untuk seleksi bersarang (E6b)
BUDGET        = 100           # evaluasi kandidat BERBEDA per run (sama dengan E5)
CHECKPOINTS   = (25, 50, 100) # titik anggaran untuk kurva konvergensi uji
N_SEED_FIN    = 5             # seed evaluasi final per konfigurasi hasil pencarian
POP           = 20
FIT_EPOCHS    = 20
FINAL_EPOCHS  = 100
SMOKE_TEST    = False         # True = versi mini untuk cek sintaks (data sintetis, epoch 1)

OUT   = '/kaggle/working' if os.path.isdir('/kaggle/working') else './out_e6'
CACHE = os.path.join(OUT, 'cache_e6')
os.makedirs(CACHE, exist_ok=True)
# pulihkan cache dari output commit sebelumnya (E6) dan dari notebook E4/E5
_n0 = 0
for f in (glob.glob('/kaggle/input/**/cache_e6/*.pkl', recursive=True)
          + glob.glob('/kaggle/input/**/cache_e45/*.pkl', recursive=True)):
    t = os.path.join(CACHE, os.path.basename(f))
    if not os.path.exists(t):
        try: shutil.copy(f, t); _n0 += 1
        except Exception: pass
print(f'cache dipulihkan: {_n0} item | total cache: {len(glob.glob(CACHE + "/*.pkl"))}')

def jam_terpakai(): return (time.time() - T_MULAI) / 3600
def sisa_jam():     return TIME_BUDGET_H - jam_terpakai()
DILEWATI = []
def cukup(tag, perlu_jam):
    if sisa_jam() > perlu_jam: return True
    DILEWATI.append(tag); print(f'[skip] {tag}: sisa {sisa_jam():.2f} jam'); return False

def ada(key): return os.path.exists(os.path.join(CACHE, key + '.pkl'))
def disk(key, fn):
    p = os.path.join(CACHE, key + '.pkl')
    if os.path.exists(p):
        with open(p, 'rb') as f: return pickle.load(f)
    v = fn(); tmp = p + f'.tmp{os.getpid()}'
    with open(tmp, 'wb') as f: pickle.dump(v, f)
    os.replace(tmp, p); return v

if SMOKE_TEST:
    N_SEED_E6C = N_SEED_SYN = N_SEED_FIN = 2; R_SEARCH = 2; R_NESTED = 1
    BUDGET = 8; CHECKPOINTS = (4, 8); POP = 4; FIT_EPOCHS = 1; FINAL_EPOCHS = 2

In [ ]:
# ============================================================================
# SEL 2. DATA SIDO, FITUR, MODEL, PELATIHAN (protokol naskah, apa adanya)
# ============================================================================
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam, RMSprop, Nadam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
print('GPU:', tf.config.list_physical_devices('GPU') or 'TIDAK ADA — nyalakan GPU T4')

def set_seed(s):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s)

TARGET, TRAIN_FRAC = 'Close', 0.8
START, END = '2019-07-01', '2025-07-01'

def _yf(tk, a=START, b=END, cols=('Open','High','Low','Close','Volume')):
    import yfinance as yf
    d = yf.download(tk, start=a, end=b, auto_adjust=True, progress=False)
    if isinstance(d.columns, pd.MultiIndex): d.columns = d.columns.get_level_values(0)
    return d.reset_index()[['Date', *cols]].dropna().reset_index(drop=True)

def muat_sido():
    if SMOKE_TEST:
        r = np.random.RandomState(0); n = 1449; c = 600*np.exp(np.cumsum(r.normal(0, .015, n)))
        return pd.DataFrame({'Date': pd.date_range(START, periods=n, freq='B'), 'Open': c, 'High': c*1.01,
                             'Low': c*.99, 'Close': c, 'Volume': r.randint(1e5, 1e6, n).astype(float)})
    try:
        d = _yf('SIDO.JK'); print('SIDO: OHLCV dari yfinance'); return d
    except Exception as ex:
        print('yfinance gagal:', type(ex).__name__)
    f = (glob.glob('/kaggle/input/**/SIDO*.xlsx', recursive=True) + glob.glob('/kaggle/input/**/SIDO*.csv', recursive=True))
    if not f: raise SystemExit('Data SIDO tidak ditemukan: nyalakan Internet atau unggah SIDO_Stock_Data.xlsx')
    d = pd.read_excel(f[0]) if f[0].endswith('xlsx') else pd.read_csv(f[0])
    d.columns = [str(c).strip().title() for c in d.columns]; d['Date'] = pd.to_datetime(d['Date'])
    print('PERINGATAN: hanya Close dari', f[0], '— angka tidak sebanding dengan set 16 fitur')
    return d.sort_values('Date').reset_index(drop=True)

def _ema(s, span): return s.ewm(span=span, adjust=False).mean()
def add_technical(df):
    d = df.copy(); c = d.Close
    d['SMA10'] = c.rolling(10).mean(); d['SMA20'] = c.rolling(20).mean()
    d['EMA10'] = _ema(c, 10); d['EMA20'] = _ema(c, 20)
    delta = c.diff(); up = delta.clip(lower=0); dn = -delta.clip(upper=0)
    rs = up.ewm(alpha=1/14, adjust=False).mean() / dn.ewm(alpha=1/14, adjust=False).mean()
    d['RSI14'] = 100 - 100/(1+rs)
    macd = _ema(c, 12) - _ema(c, 26); d['MACD'] = macd; d['MACD_sig'] = _ema(macd, 9)
    d['ret1'] = c.pct_change(); d['vol10'] = d['ret1'].rolling(10).std()
    return d

def attach_external(df):
    d = df.copy()
    for name, tk in {'JKSE': '^JKSE', 'USDIDR': 'USDIDR=X'}.items():
        try:
            if SMOKE_TEST: raise RuntimeError
            e = _yf(tk, cols=('Close',)).rename(columns={'Close': name})
            d = d.merge(e, on='Date', how='left'); d[name] = d[name].ffill().bfill()
        except Exception:
            d[name] = 0.0; print(f'PERINGATAN: {name} diisi nol')
    return d

raw = muat_sido()
df_primary = attach_external(add_technical(raw)).dropna().reset_index(drop=True)
OHLCV = [c for c in ('Open','High','Low','Close','Volume') if c in raw.columns]
TA    = ['SMA10','SMA20','EMA10','EMA20','RSI14','MACD','MACD_sig','ret1','vol10']
EXT   = [c for c in ('JKSE','USDIDR') if df_primary[c].abs().sum() > 0]
DEFAULT_FEATURES = OHLCV + TA + EXT
print(f'SIDO: {len(df_primary)} baris, {len(DEFAULT_FEATURES)} fitur')

def make_sequences(X, y, ts):
    return (np.array([X[i-ts:i] for i in range(ts, len(X))]), np.array([y[i] for i in range(ts, len(X))]))

def prepare_data(df, cols, ts):
    dX = df[cols].values.astype('float32'); dy = df[[TARGET]].values.astype('float32')
    n_tr = int(len(df)*TRAIN_FRAC)
    sx = MinMaxScaler().fit(dX[:n_tr]); sy = MinMaxScaler().fit(dy[:n_tr])
    Xs, ys = make_sequences(sx.transform(dX), sy.transform(dy), ts); sp = n_tr - ts
    return Xs[:sp], ys[:sp], Xs[sp:], ys[sp:], sy

OPTS = ['adam', 'rmsprop', 'nadam']
def _opt(name, lr): return {'adam': Adam, 'rmsprop': RMSprop, 'nadam': Nadam}[name](lr)

def build_model(shape, hp, arch='lstm'):
    K.clear_session(); inp = Input(shape=shape); x = inp
    L = int(hp.get('n_layers', 1)); Cell = GRU if arch == 'gru' else LSTM
    for i in range(L):
        x = Cell(int(hp['units']), return_sequences=(i < L-1))(x)
        if hp.get('dropout', 0) > 0: x = Dropout(hp['dropout'])(x)
    m = Model(inp, Dense(1, activation='linear')(x))
    m.compile(loss='mse', optimizer=_opt(hp.get('optimizer', 'adam'), hp.get('lr', 1e-3))); return m

def train_pred(hp, seed, df=None, feats=None, arch='lstm'):
    """Protokol final naskah: 100 epoch, ES patience 10 pada 20% akhir porsi latih.
       Kembalikan prediksi uji, nilai aktual, dan prakiraan random walk (sejajar)."""
    df = df_primary if df is None else df; feats = DEFAULT_FEATURES if feats is None else feats
    set_seed(seed)
    Xtr, ytr, Xte, yte, sy = prepare_data(df, feats, int(hp['timestep']))
    m = build_model(Xtr.shape[1:], hp, arch)
    m.fit(Xtr, ytr, validation_split=0.2, shuffle=False, epochs=FINAL_EPOCHS, batch_size=int(hp['batch_size']),
          callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)], verbose=0)
    pred = sy.inverse_transform(m.predict(Xte, verbose=0)).ravel()
    y = df[TARGET].values.astype('float64'); n_tr = int(len(df)*TRAIN_FRAC)
    return dict(pred=pred, y=y[n_tr:], rw=y[n_tr-1:-1])

def mape(y, p): return float(np.mean(np.abs((y-p)/y))*100)
def mase(y, p, y_train): return float(np.mean(np.abs(y-p)) / np.mean(np.abs(np.diff(y_train))))

n_tr0 = int(len(df_primary)*TRAIN_FRAC)
NAIVE = mape(df_primary.Close.values[n_tr0:], df_primary.Close.values[n_tr0-1:-1])
print(f'MAPE random walk SIDO: {NAIVE:.3f}')

# konfigurasi naskah (Tabel I) dan E4
STD_HP = dict(units=64,  n_layers=1, dropout=0.2,   lr=1e-3,     batch_size=32, timestep=20, optimizer='adam')
GA_HP  = dict(units=126, n_layers=1, dropout=0.140, lr=5.261e-4, batch_size=61, timestep=22, optimizer='rmsprop')
PSO_HP = dict(units=160, n_layers=1, dropout=0.1,   lr=1e-3,     batch_size=16, timestep=21, optimizer='nadam')
INT_HP = dict(units=88,  n_layers=1, dropout=0.3,   lr=3.2e-4,   batch_size=64, timestep=20, optimizer='adam')
GRU_HP = dict(units=64,  n_layers=1, dropout=0.2,   lr=1e-3,     batch_size=32, timestep=20, optimizer='adam')

In [ ]:
# ============================================================================
# SEL 3. ALAT BANTU: uji DM (HLN), Holm, ruang pencarian, fitness, GA, PSO
# ============================================================================
def dm_test(e_model, e_base, h=1):
    """DM dengan loss absolut, dua sisi, koreksi Harvey-Leybourne-Newbold.
       DM > 0 berarti model pertama (learned) LEBIH BURUK dari baseline."""
    d = np.abs(e_model) - np.abs(e_base); n = len(d); dbar = d.mean()
    gam = [np.sum((d[k:]-dbar)*(d[:n-k]-dbar))/n for k in range(h)]
    var = (gam[0] + 2*sum(gam[1:])) / n
    dm = dbar / math.sqrt(var) * math.sqrt((n + 1 - 2*h + h*(h-1)/n) / n)
    return float(dm), float(2*stats.t.sf(abs(dm), df=n-1))

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for r, i in enumerate(o): run = max(run, min(1.0, (m-r)*p[i])); adj[i] = run
    return adj

SPACES = {
  'original': dict(units=(16,160), n_layers=(1,3), dropout=(0.10,0.50), lr=(1e-4,1e-3), batch_size=(16,128), timestep=(10,30)),
  'tight':    dict(units=(32, 96), n_layers=(1,2), dropout=(0.20,0.45), lr=(1e-4,5e-4), batch_size=(32, 96), timestep=(15,25)),
}
def bounds(sp):
    s = SPACES[sp]
    lb = [s['units'][0], s['n_layers'][0], s['dropout'][0], math.log10(s['lr'][0]), s['batch_size'][0], s['timestep'][0], 0]
    ub = [s['units'][1], s['n_layers'][1], s['dropout'][1], math.log10(s['lr'][1]), s['batch_size'][1], s['timestep'][1], len(OPTS)-1e-9]
    return np.array(lb, float), np.array(ub, float)

def vec_to_hp(v):   # identik dengan E5
    return dict(units=int(round(v[0])), n_layers=int(round(v[1])), dropout=round(float(v[2]), 3), lr=float(10**v[3]),
                batch_size=int(round(v[4])), timestep=int(round(v[5])), optimizer=OPTS[int(min(int(v[6]), len(OPTS)-1))])
def cfg_key(v):     # identik dengan E5 -> cache fitness E5 terpakai ulang
    h = vec_to_hp(v)
    return (h['units'], h['n_layers'], round(h['dropout'], 2), round(math.log10(h['lr']), 2),
            h['batch_size'], h['timestep'], h['optimizer'])
def key_to_hp(k):
    return dict(units=int(k[0]), n_layers=int(k[1]), dropout=float(k[2]), lr=float(10**k[3]),
                batch_size=int(k[4]), timestep=int(k[5]), optimizer=k[6])
def key_str(k): return '_'.join(map(str, k))

def on_bounds(hp, sp):
    s = SPACES[sp]; out = []
    for kk in ('units', 'dropout', 'lr', 'batch_size', 'timestep'):
        lo, hi, v = s[kk][0], s[kk][1], float(hp[kk])
        if kk == 'lr': lo, hi, v = math.log10(lo), math.log10(hi), math.log10(v)
        if min(abs(v-lo), abs(v-hi)) <= 0.01*(hi-lo): out.append(kk)
    return ','.join(out) or 'none'

def fitness(k, mode):
    """mode 'shared' : latih 0-80% porsi latih, nilai 80-100% (blok yang JUGA dipakai early stopping) = protokol naskah/E5
       mode 'nested' : latih 0-60%, nilai 60-80%; blok 80-100% hanya untuk early stopping final."""
    hp = key_to_hp(k)
    def _hit():
        set_seed(SEED)
        Xtr, ytr, _, _, _ = prepare_data(df_primary, DEFAULT_FEATURES, hp['timestep'])
        n = len(Xtr); b = int(0.2*n)
        if mode == 'shared': Xa, ya, Xv, yv = Xtr[:n-b], ytr[:n-b], Xtr[n-b:], ytr[n-b:]
        else:                Xa, ya, Xv, yv = Xtr[:n-2*b], ytr[:n-2*b], Xtr[n-2*b:n-b], ytr[n-2*b:n-b]
        m = build_model(Xa.shape[1:], hp)
        m.fit(Xa, ya, epochs=FIT_EPOCHS, shuffle=False, batch_size=hp['batch_size'], verbose=0)
        return float(np.mean((m.predict(Xv, verbose=0).ravel() - yv.ravel())**2))
    return disk(('fit_' if mode == 'shared' else 'fitN_') + key_str(k), _hit)

class Habis(Exception): pass
class Evaluator:
    """Menghitung evaluasi BERBEDA; mencatat (best_f, best_key) setelah tiap evaluasi baru."""
    def __init__(self, mode): self.memo, self.hist, self.mode = {}, [], mode
    def __call__(self, v):
        k = cfg_key(v)
        if k not in self.memo:
            if sisa_jam() < 0.4 and not ada(('fit_' if self.mode == 'shared' else 'fitN_') + key_str(k)):
                raise Habis()
            self.memo[k] = fitness(k, self.mode)
            bk = min(self.memo, key=self.memo.get); self.hist.append((self.memo[bk], bk))
        return self.memo[k]
    @property
    def n(self): return len(self.memo)

def run_ga(lb, ub, seed, mode, budget=BUDGET, pop=POP):        # sama dengan E5
    rng = np.random.RandomState(seed); F = Evaluator(mode)
    X = rng.uniform(lb, ub, size=(pop, len(lb))); f = np.array([F(x) for x in X])
    n_elite = max(2, pop//4); stall = 0
    while F.n < budget and stall < 50:
        before = F.n; order = np.argsort(f); newX = [X[i].copy() for i in order[:n_elite]]
        while len(newX) < pop:
            a, b = [X[rng.choice(order[:pop//2])] for _ in range(2)]
            ch = np.where(rng.rand(len(lb)) < 0.5, a, b)
            mut = rng.rand(len(lb)) < 0.2
            ch = np.where(mut, ch + rng.normal(0, 0.1, len(lb))*(ub-lb), ch)
            newX.append(np.clip(ch, lb, ub))
        X = np.array(newX); f = np.array([F(x) for x in X])
        stall = stall + 1 if F.n == before else 0
    return F

def run_pso(lb, ub, seed, mode, budget=BUDGET, swarm=POP, w=0.729, c1=1.49445, c2=1.49445):   # sama dengan E5
    rng = np.random.RandomState(seed); F = Evaluator(mode)
    X = rng.uniform(lb, ub, size=(swarm, len(lb))); V = np.zeros_like(X)
    pb, pbf = X.copy(), np.full(swarm, np.inf); gb, gbf = None, np.inf
    for i in range(swarm):
        f = F(X[i]); pbf[i] = f
        if f < gbf: gbf, gb = f, X[i].copy()
    stall = it = 0
    while F.n < budget and it < 500:
        it += 1; before = F.n
        r1, r2 = rng.rand(*X.shape), rng.rand(*X.shape)
        V = w*V + c1*r1*(pb-X) + c2*r2*(gb-X); X = np.clip(X+V, lb, ub)
        for i in range(swarm):
            f = F(X[i])
            if f < pbf[i]: pbf[i], pb[i] = f, X[i].copy()
            if f < gbf: gbf, gb = f, X[i].copy()
        if F.n == before:
            stall += 1
            if stall >= 15:
                k = max(1, swarm//4); worst = np.argsort(pbf)[-k:]
                X[worst] = rng.uniform(lb, ub, size=(k, len(lb))); V[worst] = 0; pbf[worst] = np.inf; stall = 0
        else: stall = 0
    return F

def search(sp, opt, mode, r):
    """Satu run pencarian, di-cache utuh. Run 0 memakai seed 42 = run E5 (untuk ruang tight)."""
    tag = f'srch_{sp}_{opt}_{mode}_r{r}'
    if ada(tag): return disk(tag, None)
    lb, ub = bounds(sp); seed = SEED + 1000*r
    try:
        F = (run_ga if opt == 'GA' else run_pso)(lb, ub, seed, mode)
    except Habis:
        print(f'  [jeda] {tag}: anggaran habis; evaluasi yang sudah ada tersimpan, lanjut commit berikutnya')
        DILEWATI.append(tag); return None
    res = dict(hist=F.hist, n=F.n, best_key=F.hist[-1][1], best_f=F.hist[-1][0])
    return disk(tag, lambda: res)

def incumbent_at(res, b):
    """Konfigurasi terbaik setelah b evaluasi berbeda."""
    return res['hist'][min(b, len(res['hist'])) - 1][1]

def final_eval(k, n_seed=N_SEED_FIN):
    """Evaluasi uji final (protokol naskah) untuk satu konfigurasi; di-cache per seed."""
    hp = key_to_hp(k); vals = []
    for s in range(n_seed):
        key = f'fin_{key_str(k)}_s{s}'
        if not ada(key) and sisa_jam() < 0.3: break
        r = disk(key, lambda s=s: train_pred(hp, SEED + s)); vals.append(mape(r['y'], r['pred']))
    return np.array(vals, float)
print('alat bantu siap')

In [ ]:
# ============================================================================
# SEL 4. E6c — uji DM semua model terlatih vs random walk (Reviewer B3, B5)
#   DM dihitung pada prakiraan rata-rata 15 seed (ensemble) dan per seed.
#   Holm dikoreksi antar-model. GRU = model terlatih dengan MAPE terendah di Tabel I.
# ============================================================================
E6C = pd.DataFrame()
if RUN['E6C'] and cukup('E6c', 0.5):
    MODELS = [('GRU (default)', GRU_HP, 'gru'), ('LSTM (default)', STD_HP, 'lstm'),
              ('LSTM-GA (Table I)', GA_HP, 'lstm'), ('LSTM-PSO (Table I)', PSO_HP, 'lstm'),
              ('LSTM (interior)', INT_HP, 'lstm')]
    # konfigurasi GA/PSO-tight dari cache E5 bila tersedia
    for nm, lbl in (('search_ga_tight', 'LSTM-GA (tight)'), ('search_pso_tight', 'LSTM-PSO (tight)')):
        if ada(nm):
            v = disk(nm, None)[0]; MODELS.append((lbl, vec_to_hp(v), 'lstm')); print(lbl, vec_to_hp(v))
    y_train = df_primary.Close.values[:n_tr0]
    rows = []
    for lbl, hp, arch in MODELS:
        outs = []
        for s in range(N_SEED_E6C):
            key = f'e6c_{arch}_{key_str(cfg_key([hp["units"], hp.get("n_layers",1), hp["dropout"], math.log10(hp["lr"]), hp["batch_size"], hp["timestep"], OPTS.index(hp["optimizer"])]))}_s{s}'
            if not ada(key) and sisa_jam() < 0.3: break
            outs.append(disk(key, lambda hp=hp, s=s, arch=arch: train_pred(hp, SEED + s, arch=arch)))
        if not outs: continue
        y, rw = outs[0]['y'], outs[0]['rw']
        per = [dm_test(y - o['pred'], y - rw) for o in outs]
        pm = np.mean([o['pred'] for o in outs], 0); dm, p = dm_test(y - pm, y - rw)
        mp = np.array([mape(y, o['pred']) for o in outs])
        rows.append(dict(model=lbl, n_seed=len(outs), MAPE_mean=mp.mean(), MAPE_sd=mp.std(ddof=1) if len(mp) > 1 else np.nan,
                         MAPE_ensemble=mape(y, pm), MASE_ensemble=mase(y, pm, y_train), DM_ensemble=dm, p_raw=p,
                         seeds_RW_sig_better=np.mean([(d > 0 and q < .05) for d, q in per]),
                         seeds_model_sig_better=np.mean([(d < 0 and q < .05) for d, q in per])))
        print(f'  {lbl:20s} MAPE {mp.mean():.3f}  DM {dm:+.2f}  p {p:.4g}')
    E6C = pd.DataFrame(rows)
    if len(E6C):
        E6C['p_holm'] = holm(E6C.p_raw)
        E6C['verdict'] = np.where((E6C.p_holm < .05) & (E6C.DM_ensemble > 0), 'RW better',
                          np.where((E6C.p_holm < .05) & (E6C.DM_ensemble < 0), 'model better', 'n.s.'))
        E6C.round(4).to_csv(f'{OUT}/TableE6c_DM_all_models.csv', index=False)
        print('\n' + E6C.round(4).to_string(index=False))
        print(f'RW MAPE = {NAIVE:.3f}. DM > 0 berarti random walk lebih akurat.')

In [ ]:
# ============================================================================
# SEL 5. E6d — baterai deret sintetis (Reviewer B6)
#   3 deret TAK terprediksi  -> kerangka harus TIDAK menyatakan model unggul
#   3 deret TERPREDIKSI      -> kerangka harus menyatakan model unggul (DM < 0, signifikan)
#   Close saja, konfigurasi interior, N_SEED_SYN seed, 3 realisasi (seed data) per deret.
# ============================================================================
def synth(kind, n=1449, seed=7):
    r = np.random.RandomState(seed); t = np.arange(n)
    if kind == 'Random walk':            x = np.cumsum(r.normal(0, .015, n))
    elif kind == 'Random walk + drift':  x = np.cumsum(r.normal(.0008, .015, n))
    elif kind == 'GARCH random walk':
        e = np.zeros(n); s2 = np.full(n, 2e-4)
        for i in range(1, n):
            s2[i] = 1e-5 + .10*e[i-1]**2 + .85*s2[i-1]; e[i] = r.normal(0, math.sqrt(s2[i]))
        x = np.cumsum(e)
    elif kind == 'AR(1) returns, phi=0.3':
        e = np.zeros(n)
        for i in range(1, n): e[i] = .3*e[i-1] + r.normal(0, .01)
        x = np.cumsum(e)
    elif kind == 'Mean-reverting level, phi=0.6':
        x = np.zeros(n)
        for i in range(1, n): x[i] = .6*x[i-1] + r.normal(0, .02)
    elif kind == 'Seasonal + AR':
        a = np.zeros(n)
        for i in range(1, n): a[i] = .3*a[i-1] + r.normal(0, .005)
        x = .08*np.sin(2*np.pi*t/5) + .05*np.sin(2*np.pi*t/21) + a
    return pd.DataFrame({'Date': pd.date_range(START, periods=n, freq='B'), 'Close': 100*np.exp(x)})

SYN = {'Random walk': 'no skill', 'Random walk + drift': 'no skill', 'GARCH random walk': 'no skill',
       'AR(1) returns, phi=0.3': 'skill', 'Mean-reverting level, phi=0.6': 'skill', 'Seasonal + AR': 'skill'}
N_REAL = 1 if SMOKE_TEST else 3

def eval_series(tag, df, n_seed=N_SEED_SYN, hp=INT_HP):
    outs = []
    for s in range(n_seed):
        key = f'{tag}_s{s}'
        if not ada(key) and sisa_jam() < 0.3: break
        outs.append(disk(key, lambda s=s: train_pred(hp, SEED + s, df=df, feats=['Close'])))
    if not outs: return None
    y, rw = outs[0]['y'], outs[0]['rw']; pm = np.mean([o['pred'] for o in outs], 0)
    dm, p = dm_test(y - pm, y - rw); y_tr = df.Close.values[:int(len(df)*TRAIN_FRAC)]
    return dict(n_seed=len(outs), MAPE_model=mape(y, pm), MAPE_RW=mape(y, rw),
                MASE=mase(y, pm, y_tr), DM=dm, p=p)

E6D = pd.DataFrame()
if RUN['E6D'] and cukup('E6d', 0.4):
    rows = []
    for kind, exp in SYN.items():
        for rz in range(N_REAL):
            r_ = eval_series(f'syn_{kind.replace(" ", "").replace(",", "").replace("=", "")}_d{rz}', synth(kind, seed=7+rz))
            if r_: rows.append(dict(series=kind, realization=rz, expected=exp, **r_))
    E6D = pd.DataFrame(rows)
    if len(E6D):
        E6D['verdict'] = np.where((E6D.p < .05) & (E6D.DM < 0), 'model better',
                          np.where((E6D.p < .05) & (E6D.DM > 0), 'RW better', 'n.s.'))
        E6D['correct'] = (E6D.expected == 'skill') == (E6D.verdict == 'model better')
        E6D.round(4).to_csv(f'{OUT}/TableE6d_synthetic.csv', index=False)
        print(E6D.round(3).to_string(index=False))
        print('\nRingkasan per deret:')
        print(E6D.groupby(['series', 'expected']).agg(MASE=('MASE', 'mean'), DM=('DM', 'mean'),
              correct=('correct', 'mean')).round(3).to_string())

In [ ]:
# ============================================================================
# SEL 6. E6e — pasar dan periode lain (Reviewer A11). Perlu Internet ON.
#   Protokol sama dengan E6d (Close saja, konfigurasi interior). Sebagai acuan,
#   SIDO dengan Close saja juga dihitung agar sebanding.
# ============================================================================
EXT_SETS = [('SIDO.JK', '2019-07-01', '2025-07-01'),
            ('^GSPC', '2019-07-01', '2025-07-01'), ('^N225', '2019-07-01', '2025-07-01'),
            ('BTC-USD', '2019-07-01', '2025-07-01'), ('EURUSD=X', '2019-07-01', '2025-07-01'),
            ('^GSPC', '2012-07-01', '2018-07-01'), ('^JKSE', '2012-07-01', '2018-07-01')]
E6E = pd.DataFrame()
if RUN['E6E'] and not SMOKE_TEST and cukup('E6e', 0.4):
    rows = []
    for tk, a, b in EXT_SETS:
        try:
            d = _yf(tk, a, b, cols=('Close',))
        except Exception as ex:
            print('[gagal unduh]', tk, type(ex).__name__); continue
        r_ = eval_series(f'mkt_{tk.replace("^", "").replace("=", "")}_{a[:4]}', d)
        if r_: rows.append(dict(series=tk, period=f'{a[:4]}–{b[:4]}', n_obs=len(d), **r_)); print(rows[-1])
    E6E = pd.DataFrame(rows)
    if len(E6E):
        E6E['p_holm'] = holm(E6E.p)
        E6E['verdict'] = np.where((E6E.p_holm < .05) & (E6E.DM > 0), 'RW better',
                          np.where((E6E.p_holm < .05) & (E6E.DM < 0), 'model better', 'n.s.'))
        E6E.round(4).to_csv(f'{OUT}/TableE6e_cross_market.csv', index=False)
        print(E6E.round(3).to_string(index=False))

In [ ]:
# ============================================================================
# SEL 7. E6a — pencarian berulang + konvergensi vs anggaran (Reviewer B2)
#   R_SEARCH run independen untuk GA dan PSO di ruang 'original' (naskah) dan
#   'tight' (E5). Untuk tiap run, konfigurasi terbaik setelah 25, 50, dan 100
#   evaluasi dievaluasi pada data uji (N_SEED_FIN seed) -> kurva konvergensi
#   dalam satuan validasi DAN uji. Run 0 di ruang tight = run E5.
# ============================================================================
def jalankan_pencarian(spaces, mode, R):
    rows = []
    for sp in spaces:
        for opt in ('GA', 'PSO'):
            for r in range(R):
                tag = f'srch_{sp}_{opt}_{mode}_r{r}'
                if not ada(tag) and not cukup(tag, 0.6): continue
                res = search(sp, opt, mode, r)
                if res is None: continue
                for b in CHECKPOINTS:
                    k = incumbent_at(res, b); hp = key_to_hp(k)
                    a = final_eval(k)
                    rows.append(dict(space=sp, optimizer=opt, selection=mode, run=r, budget=b,
                                     distinct_evals=res['n'], best_val_mse=res['hist'][min(b, len(res['hist']))-1][0],
                                     on_bounds=on_bounds(hp, sp), n_seed=len(a),
                                     test_MAPE_mean=a.mean() if len(a) else np.nan,
                                     test_MAPE_sd=a.std(ddof=1) if len(a) > 1 else np.nan,
                                     **{f'hp_{kk}': vv for kk, vv in hp.items()}))
                last = rows[-1] if rows and rows[-1]['run'] == r and rows[-1]['optimizer'] == opt else None
                if last: print(f"  {tag}: val {last['best_val_mse']:.6f} | uji {last['test_MAPE_mean']:.3f}% | tepi: {last['on_bounds']}")
    return pd.DataFrame(rows)

E6A = jalankan_pencarian(('original', 'tight'), 'shared', R_SEARCH) if RUN['E6A'] else pd.DataFrame()
if len(E6A):
    E6A.round(5).to_csv(f'{OUT}/TableE6a_repeated_search.csv', index=False)
    fin = E6A[E6A.budget == max(CHECKPOINTS)]
    S = (fin.groupby(['space', 'optimizer'])
            .agg(runs=('run', 'count'), test_MAPE_mean=('test_MAPE_mean', 'mean'), test_MAPE_sd_runs=('test_MAPE_mean', 'std'),
                 test_MAPE_min=('test_MAPE_mean', 'min'), test_MAPE_max=('test_MAPE_mean', 'max'),
                 share_runs_on_bounds=('on_bounds', lambda s: float((s != 'none').mean())))
            .reset_index())
    S['vs_naive'] = S.test_MAPE_mean / NAIVE
    tests = []
    for sp in S.space.unique():
        g = fin[(fin.space == sp) & (fin.optimizer == 'GA')].test_MAPE_mean.dropna()
        q = fin[(fin.space == sp) & (fin.optimizer == 'PSO')].test_MAPE_mean.dropna()
        p = stats.mannwhitneyu(g, q, alternative='two-sided').pvalue if len(g) > 1 and len(q) > 1 else np.nan
        tests.append(dict(space=sp, GA_vs_PSO_mannwhitney_p=p))
    S = S.merge(pd.DataFrame(tests), on='space')
    S.round(4).to_csv(f'{OUT}/TableE6a_summary.csv', index=False)
    print('\n' + S.round(3).to_string(index=False))
    B = (E6A.groupby(['space', 'optimizer', 'budget']).test_MAPE_mean.agg(['mean', 'std', 'count']).round(3))
    B.to_csv(f'{OUT}/TableE6a_budget.csv'); print('\nMAPE uji vs anggaran evaluasi:\n' + B.to_string())

    # Gambar: (a,b) best validation MSE vs evaluasi (median & IQR antar-run); (c) MAPE uji vs anggaran
    import matplotlib.ticker as mticker
    fig, axs = plt.subplots(1, 3, figsize=(7.2, 2.8))
    for ax, sp in zip(axs[:2], ('original', 'tight')):
        for opt, ls in (('GA', '-'), ('PSO', '--')):
            H = [disk(f'srch_{sp}_{opt}_shared_r{r}', None)['hist'] for r in range(R_SEARCH) if ada(f'srch_{sp}_{opt}_shared_r{r}')]
            if not H: continue
            L = max(len(h) for h in H); M = np.array([[x[0] for x in h] + [h[-1][0]]*(L-len(h)) for h in H])
            x = np.arange(1, L+1)
            ax.plot(x, np.median(M, 0), ls, color='k', lw=1.3, label=f'{opt} (n={len(H)})')
            ax.fill_between(x, np.percentile(M, 25, 0), np.percentile(M, 75, 0), color='0.6', alpha=.35, lw=0)
        ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0)); ax.yaxis.get_offset_text().set_fontsize(7)
        ax.set_xlabel('Distinct evaluations', fontsize=8); ax.set_ylabel('Best validation MSE', fontsize=8)
        ax.set_title(f'({"a" if sp == "original" else "b"}) {sp} space', loc='left', fontsize=8)
        ax.legend(frameon=False, fontsize=7); ax.tick_params(labelsize=7)
    ax = axs[2]
    for (sp, opt), g in E6A.groupby(['space', 'optimizer']):
        m = g.groupby('budget').test_MAPE_mean.agg(['mean', 'min', 'max'])
        ax.errorbar(m.index + (0 if opt == 'GA' else 2), m['mean'], yerr=[m['mean']-m['min'], m['max']-m['mean']],
                    fmt='o-' if sp == 'original' else 's--', color='k' if opt == 'GA' else '0.5', ms=4, lw=1, capsize=2,
                    label=f'{opt}, {sp}')
    ax.axhline(NAIVE, color='k', lw=1); ax.text(0.02, NAIVE, f'Random walk ({NAIVE:.2f})', transform=ax.get_yaxis_transform(), va='bottom', fontsize=6.5)
    ax.set_xlabel('Evaluation budget', fontsize=8); ax.set_ylabel('Test MAPE (%)', fontsize=8)
    ax.set_title('(c) Test MAPE vs budget', loc='left', fontsize=8); ax.legend(frameon=False, fontsize=6); ax.tick_params(labelsize=7)
    fig.tight_layout(); fig.savefig(f'{OUT}/FigE6a_convergence.png', dpi=600, bbox_inches='tight'); plt.show()

In [ ]:
# ============================================================================
# SEL 8. E6b — seleksi bersarang (Reviewer B4)
#   Fitness dihitung pada blok 60-80% porsi latih; blok 80-100% hanya untuk
#   early stopping final. Dibandingkan dengan seleksi 'shared' (run yang sama di E6a).
# ============================================================================
E6B = jalankan_pencarian(('tight',), 'nested', R_NESTED) if RUN['E6B'] else pd.DataFrame()
if len(E6B):
    E6B.round(5).to_csv(f'{OUT}/TableE6b_nested.csv', index=False)
    both = pd.concat([E6A[(E6A.space == 'tight') & (E6A.run < R_NESTED)] if len(E6A) else pd.DataFrame(), E6B])
    both = both[both.budget == max(CHECKPOINTS)]
    cmp_ = both.groupby(['optimizer', 'selection']).test_MAPE_mean.agg(['mean', 'std', 'min', 'max', 'count']).round(3)
    cmp_.to_csv(f'{OUT}/TableE6b_shared_vs_nested.csv'); print(cmp_.to_string())

In [ ]:
# ============================================================================
# SEL 10. Tambahan E6d — pembanding AR linear (oracle) & pergeseran rentang harga
#   Dipakai di naskah (Bagian III-E): (i) AR(1) yang dispesifikasi benar hanya
#   signifikan pada 1 dari 3 realisasi "AR(1) returns" -> strukturnya lemah;
#   (ii) korelasi Spearman antara porsi harga uji di luar rentang latih dan MASE LSTM.
#   Murni numpy, tidak melatih jaringan (beberapa detik).
# ============================================================================
rows = []
for kind in SYN:
    for rz in range(N_REAL):
        df = synth(kind, seed=7 + rz); y = df.Close.values; n = len(y); ntr = int(n*TRAIN_FRAC)
        yt, rw, ly = y[ntr:], y[ntr-1:-1], np.log(y)
        if kind.startswith('Seasonal'):
            L = 21; X = np.array([ly[i-L:i] for i in range(L, n)]); Y = ly[L:]; tr = np.arange(L, n) < ntr
            A = np.c_[X, np.ones(len(X))]; w = np.linalg.lstsq(A[tr], Y[tr], rcond=None)[0]; pred = np.exp(A @ w)[~tr]
        elif kind.startswith('Mean-reverting'):
            x0 = ly[:ntr]; w = np.linalg.lstsq(np.c_[x0[:-1], np.ones(ntr-1)], x0[1:], rcond=None)[0]
            pred = np.exp(w[0]*ly[ntr-1:-1] + w[1])
        else:
            r = np.diff(ly); r0 = r[:ntr-1]
            w = np.linalg.lstsq(np.c_[r0[:-1], np.ones(len(r0)-1)], r0[1:], rcond=None)[0]
            pred = np.exp(ly[ntr-1:-1] + w[0]*r[ntr-2:-1] + w[1])
        d_, p_ = dm_test(yt - pred, yt - rw)
        rows.append(dict(series=kind, realization=rz, oracle_MASE=mase(yt, pred, y[:ntr]), oracle_DM=d_, oracle_p=p_,
                         share_test_outside_train=np.mean((yt > y[:ntr].max()) | (yt < y[:ntr].min()))))
ORA = pd.DataFrame(rows)
if len(E6D):
    ORA = E6D.merge(ORA, on=['series', 'realization'])
    print('Spearman(share outside range, LSTM MASE):', stats.spearmanr(ORA.share_test_outside_train, ORA.MASE))
ORA.round(4).to_csv(f'{OUT}/TableE6d_oracle_range.csv', index=False)
print(ORA.round(3).to_string(index=False))

In [ ]:
# ============================================================================
# SEL 9. RINGKASAN UNTUK NASKAH
# ============================================================================
def ringkas():
    print(f'\nRandom walk SIDO MAPE = {NAIVE:.3f}%')
    if len(E6C):
        print('\n[E6c] DM vs random walk (Holm antar-model):')
        print(E6C[['model', 'MAPE_mean', 'DM_ensemble', 'p_holm', 'verdict']].round(4).to_string(index=False))
    if len(E6D):
        print(f"\n[E6d] Sintetis: {int(E6D.correct.sum())}/{len(E6D)} keputusan sesuai harapan")
    if len(E6E):
        print('\n[E6e] Lintas pasar:'); print(E6E[['series', 'period', 'MAPE_model', 'MAPE_RW', 'MASE', 'DM', 'p_holm', 'verdict']].round(3).to_string(index=False))
    if len(E6A):
        print('\n[E6a] lihat TableE6a_summary.csv dan FigE6a_convergence.png')
    if len(E6B):
        print('\n[E6b] lihat TableE6b_shared_vs_nested.csv')
ringkas()
print(f'\nTerpakai {jam_terpakai():.2f} jam dari {TIME_BUDGET_H} jam')
print('Dilewati commit ini:', DILEWATI or 'tidak ada')
print('Isi cache:', len(glob.glob(CACHE + '/*.pkl')), 'item')